In [ ]:
#Last Edited: 2025/06/18
#File changed more recently (2025/11/18) to commit (with Cell outputs deleted)

In [ ]:
#This was my starting point 

#This is example code from chatGPT, I want to take my annotations, and my mat files, 
#and create 24 hour files, with an embedded annotations dataset

#I'm using the "science_env" kernel

In [43]:
#PART 1: CONVERT MAT to h5

import os
import scipy.io
import numpy as np
import h5py

# Helper function to convert MATLAB structs to Python dicts
def matlab_struct_to_dict(matobj):
    d = {}
    for fieldname in matobj._fieldnames:
        value = getattr(matobj, fieldname)
        if isinstance(value, np.ndarray) and value.dtype.names:
            # Nested struct array
            d[fieldname] = [matlab_struct_to_dict(el) for el in value]
        elif hasattr(value, '_fieldnames'):
            d[fieldname] = matlab_struct_to_dict(value)
        else:
            d[fieldname] = value
    return d

# 2. Preview each field (meta, data, units, config)
def preview_struct(struct, name):
    print(f"\n== {name.upper()} ==")
    if hasattr(struct, '_fieldnames'):
        for fname in struct._fieldnames:
            value = getattr(struct, fname)
            if isinstance(value, np.ndarray):
                print(f"  {fname}: array shape {value.shape}, dtype {value.dtype}")
            elif hasattr(value, '_fieldnames'):
                print(f"  {fname}: nested struct")
            else:
                print(f"  {fname}: {type(value)} ({value})")
    else:
        print(struct)


def save_dict(h5group, d):
    #Helper function for saving the outputs
    for k, v in d.items():
        if isinstance(v, dict):
            g = h5group.create_group(k)
            save_dict(g, v)
        elif isinstance(v, np.ndarray) and v.dtype.names:  # structured array
            for name in v.dtype.names:
                h5group.create_dataset(f"{k}/{name}", data=v[name])
        else:
            try:
                h5group.create_dataset(k, data=v)
            except TypeError:
                try:
                    h5group.create_dataset(k, data=str(v))
                except Exception:
                    pass


def extract_mat_to_h5(mat_path, output_folder):
    # 1. Load the .mat file
    mat = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)

    for key in ['meta', 'data', 'units', 'config']:
        preview_struct(mat[key], key)

    # 3. Extract fields into Python objects
    meta = matlab_struct_to_dict(mat['meta'])
    data = matlab_struct_to_dict(mat['data'])
    units = matlab_struct_to_dict(mat['units'])
    config = matlab_struct_to_dict(mat['config'])

    # Quick preview main data time series
    print("\n=== DATA FIELD SAMPLES ===")
    for k, v in data.items():
        if isinstance(v, np.ndarray):
            print(f"{k}: shape {v.shape}, dtype {v.dtype}")
        else:
            print(f"{k}: {type(v)} ({v})")

    # 4. Save to HDF5 for future use
            
    # Create folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Generate new filename with .h5 extension
    filename = os.path.basename(mat_path)
    output_filename = os.path.splitext(filename)[0] + '.h5'
    h5_output_path = os.path.join(output_folder, output_filename)
    #print(f"Output path: {output_path}")
    #h5_output_filename = 'organized_data.h5'

    with h5py.File(h5_output_path, 'w') as h5f:
        save_dict(h5f.create_group('meta'), meta)
        save_dict(h5f.create_group('data'), data)
        save_dict(h5f.create_group('units'), units)
        save_dict(h5f.create_group('config'), config)

    print("\n✅ Data extracted and saved as organized_data.h5!")

In [ ]:
# Path to your .mat file

data_folder = r'F:\Documents\Projects\ADCP\scan_for_data\BACAX\ADCP2MHZ\20140401\\'
file_list = os.listdir(data_folder)
mat_files = {k for k in file_list if os.path.splitext(k)[1] == ".mat"}

print(mat_files)

mat_paths = []
for filename in mat_files:
    mat_paths.append(data_folder + filename) 

#data_folder = r'F:\\Documents\\Projects\\ADCP\\ADCP Monitoring Management\\20250422_BACAX_2MHZ\\'
#filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250422T000006Z_20250422T115956Z-binMapNearest.mat'
#mat_path = data_folder + filename

# Define output folder
output_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files'

#Run the extraction
for mat_path in mat_paths:
    extract_mat_to_h5(mat_path, output_folder) 

#extract_mat_to_h5(mat_path, output_folder) # For a single file

{'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20140401T000000Z_20140501T000000Z-Ensemble300s_binMapNearest.mat'}

== META ==
  deviceID: <class 'int'> (11302)
  creationDate: <class 'str'> (20250308T014540Z)
  deviceName: <class 'str'> (Nortek Aquadopp HR-Profiler 2 MHz 2700)
  deviceCode: <class 'str'> (BC_POD1_AD2M)
  deviceCategory: <class 'str'> (Acoustic Doppler Current Profiler 2 MHz)
  deviceCategoryCode: <class 'str'> (ADCP2MHZ)
  lat: <class 'float'> (48.3165166667)
  lon: <class 'float'> (-126.0502833333)
  depth: <class 'int'> (986)
  deviceHeading: <class 'int'> (183)
  devicePitch: <class 'float'> (nan)
  deviceRoll: <class 'float'> (nan)
  siteName: <class 'str'> (CanyonAxis_ADCP_2013-05)
  locationName: <class 'str'> (Barkley Canyon)
  stationCode: <class 'str'> (BACAX)
  dataQualityComments: array shape (0,), dtype <U1
  MobilePositionSensor: nested struct
  deploymentDateFrom: <class 'float'> (735366.7779166667)
  deploymentDateTo: <class 'float'

In [69]:
#PART 2: Split up h5 monthly files and embed annotations
import h5py
import numpy as np
import scipy.io
from datetime import datetime, timezone
from dateutil import parser as dateparse
import os

# ---- Annotation extraction and conversion from MATLAB ----
def load_matlab_annotations(mat_path):
    mat = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    ann_struct = mat['annotations']
    if ann_struct.dtype.names:  # single annotation
        ann_struct = [ann_struct]

    annotations = []
    for a in ann_struct:
        annotation_dict = {}
        annotation_dict['class'] = str(getattr(a, 'class'))
        annotation_dict['startDate'] = str(getattr(a, 'startDate'))
        annotation_dict['endDate'] = str(getattr(a, 'endDate'))
        annotation_dict['comment'] = str(getattr(a, 'comment'))
        #dct['status'] = str(getattr(a, 'status'))
        # Parse datetime (assumes UTC, adjust if needed)
        annotation_dict['start_datetime'] = dateparse.parse(annotation_dict['startDate'].replace('.0','')).replace(tzinfo=timezone.utc)
        annotation_dict['end_datetime']   = dateparse.parse(annotation_dict['endDate'].replace('.0','')).replace(tzinfo=timezone.utc)
        annotations.append(annotation_dict)
    return annotations

# ---- Splitter ----
def split_h5_to_24hr_files_with_ann(input_filename, out_dir, annotation_mat_file, time_ds_path='/data/time'):
    # Step 1: Load annotations and convert their datetimes to UNIX seconds
    annotations = load_matlab_annotations(annotation_mat_file)
    for a in annotations:
        # Convert to int seconds since epoch (UTC, for fast search)
        a['start_time_sec'] = int(a['start_datetime'].timestamp())
        a['end_time_sec'] = int(a['end_datetime'].timestamp())

    with h5py.File(input_filename, 'r') as h5in:
        # Step 2: Load main time vector & fields
        time = h5in[time_ds_path][:]
        
        if np.issubdtype(time.dtype, np.floating):  # likely MATLAB datenum
            def datenum_to_epochsecs(dn):
                return np.round((dn - 719529) * 86400).astype('int64')
            time_sec = datenum_to_epochsecs(time)
        else:
            if np.issubdtype(time.dtype, np.datetime64):
                # If HDF stores as np.datetime64, convert to int seconds
                time_sec = time.astype('datetime64[s]').astype('int64')
            else:
                time_sec = time.astype('int64')  # already in seconds

        # Now, time_sec is ALWAYS seconds (int64)
        # If you wish, also get as datetime64 for some operations:
        # time_dt64 = time_sec.astype('datetime64[s]')

        #Verify that the file is:
        # 1) At least 24 hours long
        # 2) Has a 5 minute sample interval
                
        # 1. Check duration
        dur_s = time_sec[-1] - time_sec[0]      # total duration in seconds
        if dur_s < 86400:
            raise ValueError(f"Input file spans less than 24 hours ({dur_s/3600:.2f} hours)!")

        # 2. Check (average) interval
        diffs = np.diff(time_sec)
        median_dt = np.median(diffs)
        if not np.allclose(median_dt, 300, atol=2):   # 2s tolerance for rounding
            raise ValueError(
                f"Input file does not have a 5-minute interval (median step = {median_dt} s)"
            )
        
        # Load all other /data datasets
        data_group = h5in['/data']
        #main_fields = {k: data_group[k][:] for k in data_group if k != "time"}
        

        # Step 3: Compute the split points (midnight to midnight UTC, last segment aligned at end)
        start_all = time_sec[0]
        end_all = time_sec[-1]
        #time_dt64 = time_sec.astype('datetime64[s]')
        #start_all = time_dt64[0]
        #end_all   = time_dt64[-1]

        # Use UTC midnights for the intervals
        start_day = datetime.fromtimestamp(start_all, tz=timezone.utc).replace(
            hour=0, minute=0, second=0, microsecond=0
        )
        start_day_sec = int(start_day.timestamp())
        intervals = []
        current_start = start_day_sec

        # Round start to nearest midnight after/before as appropriate
        #day0 = datetime.fromtimestamp(time_sec[0].UTC).replace(hour=0, minute=0, second=0, microsecond=0)
        #day0 = datetime.utcfromtimestamp(time_sec[0]).replace(hour=0, minute=0, second=0, microsecond=0)
        #day0 = np.datetime64(day0, 's')
        #intervals = []
        #current_start = day0

        # Iterate 24h windows, these may have overlap (start and end) but that's okay!
        
        #First, ensure that the first entry is 24 hours, even if not starting UTC midnight,
        if current_start < start_all:
            # Output a "first chunk" from start_all for 24h
            intervals.append((start_all, start_all + 86400 - 1))
            current_start += 86400

        # then do all other intervals so that they are utc midnight
        while current_start + 86400 - 1 <= end_all:
            intervals.append((current_start, current_start + 86400 - 1))
            current_start += 86400

        # Add tail if needed that is also 24 hours
        if end_all > intervals[-1][1]:
            intervals.append((end_all - 86400 + 1, end_all))

        #while current_start + np.timedelta64(86400-1, 's') <= end_all:
        #    intervals.append((current_start, current_start + np.timedelta64(86400-1, 's')))
        #    current_start += np.timedelta64(24, 'h')
        ## Add tail (if ending in middle of a day)
        #if end_all > intervals[-1][1]:
        #    intervals.append((end_all - np.timedelta64(86400-1, 's'), end_all))
        #    #intervals.append((end_all - np.timedelta64(24, 'h'), end_all))

        # Step 4: For each interval, slice data and embed annotations
        os.makedirs(out_dir, exist_ok=True)
        for segment_start, segment_end in intervals:
            #start_sec = int(segment_start.astype('int64'))
            #end_sec   = int(segment_end.astype('int64'))
            left_idx = np.searchsorted(time_sec, segment_start, side="left")
            right_idx = np.searchsorted(time_sec, segment_end, side="right")
            # Indices to slice data arrays
            # left_idx  = np.searchsorted(time_sec, start_sec, side="left")
            #right_idx = np.searchsorted(time_sec, end_sec,   side="right")
            #segment_len = right_idx - left_idx
            # Guarantee length is 24h = 288 samples (minutes) if possible, else pad last
            #required_len = end_sec - start_sec + 1
            # ---- slice and pad main arrays ----
            #Slice and pad Time
            seg_time = time_sec[left_idx:right_idx] 
            sample_dt = np.median(np.diff(time_sec))   # e.g., 300
            samples_per_window = int(np.round((segment_end - segment_start) / sample_dt))
            #samples_per_window = int(np.round((end_sec - start_sec) / sample_dt)) #e.g. 288 # + 1  # e.g., 289
            pad_len = samples_per_window - len(seg_time)
            #pad_len = int(required_len - len(seg_time))

            #Slice and pad Data - Will find whichever axis is same length as time:
            main_fields = {}
            for k in data_group:
                dset = data_group[k]
                if dset.shape == ():  # scalar
                    main_fields[k] = dset[()]  # store as is
                    continue
                # Find the axis whose length matches time
                matching_axes = [i for i, sz in enumerate(dset.shape) if sz == len(time_sec)]
                if len(matching_axes) == 1:
                    time_axis = matching_axes[0]
                    # Prepare slices: [:, :, :, ...] but time_axis gets slice(left_idx, right_idx)
                    slicer = [slice(None)] * dset.ndim
                    slicer[time_axis] = slice(left_idx, right_idx)
                    main_fields[k] = dset[tuple(slicer)]
                elif len(matching_axes) == 0:
                    # Not time dependent; save as is
                    main_fields[k] = dset[()]
                    print(f"Retaining static dataset: {k} shape {dset.shape}")
                else:
                    print(f"Warning: {k} has multiple axes matching len(time); skipping for safety")
                    # Handle as you see fit

            #seg_fields = {k: arr[left_idx:right_idx] for k, arr in main_fields.items()}
            if pad_len > 0:
                seg_time = np.pad(seg_time, (0, pad_len), 'edge')
                for k in main_fields:
                    arr = main_fields[k]
                    if isinstance(arr, np.ndarray) and arr.shape[0] == len(seg_time) - pad_len:
                        main_fields[k] = np.pad(arr, ((0, pad_len),), 'edge')

                #for k in seg_fields:
                #    seg_fields[k] = np.pad(seg_fields[k], (0, pad_len), 'edge')
            # ---- collect relevant annotation rows ----
            ann_rows = []
            for a in annotations:
                # If annotation overlaps this chunk

                #*****************************************
                if a['end_time_sec'] >= segment_start and a['start_time_sec'] < segment_end:
                    # Restrict (clip) indices to chunk bounds, relative to seg_time
                    sidx = np.searchsorted(seg_time, a['start_time_sec'], side='left')
                    eidx = np.searchsorted(seg_time, a['end_time_sec'], side='right') - 1
                    sidx = max(sidx, 0)
                    eidx = min(eidx, len(seg_time) - 1)
                    ann_rows.append((
                        a['class'].encode('utf8'),
                        a['start_time_sec'],
                        a['end_time_sec'],
                        sidx,
                        eidx,
                        a['comment'].encode('utf8'),
                        #a['status'].encode('utf8')
                    ))
            # Structured dtype for HDF5
            ann_dtype = np.dtype([
                ('class','S32'),
                ('start_time','i8'),
                ('end_time','i8'),
                ('start_index','i8'),
                ('end_index','i8'),
                ('comment','S255'),
                #('status','S16')
            ])
            ann_array = np.array(ann_rows, dtype=ann_dtype) if ann_rows else np.zeros((0,), dtype=ann_dtype)
            # ---- SAVE output ----
            outfn = os.path.join(
                out_dir, 
                f"{datetime.utcfromtimestamp(segment_start):%Y%m%dT%H%M%S}_"
                f"{datetime.utcfromtimestamp(segment_end):%Y%m%dT%H%M%S}.h5"
            )
            with h5py.File(outfn, 'w') as h5out:
                dgrp = h5out.create_group('data')
                dgrp.create_dataset("time", data=seg_time)  ### <---- Use "time" in seconds, not the MATLAB time
                for k, arr in main_fields.items():
                    if np.isscalar(arr):
                        # Save scalars as attributes
                        dgrp.attrs[k] = arr
                    elif k != "time":                       ### <---- Don't write time again
                        #Save all other variables
                        dgrp.create_dataset(k, data=arr)
                #Embed the annotations
                h5out.create_dataset("annotations", data=ann_array)
                #ann_grp = h5out.create_group('annotations')
                #ann_grp.create_dataset('events', data=ann_array)
        
            print("Wrote", outfn, f"(len={len(seg_time)}, {len(ann_array)} annotations)")



In [ ]:
# ---- Usage example ----
# split_h5_to_24hr_files_with_ann(
#     'alldata.h5',             # your big HDF5 source
#     'output_24h_h5',          # output dir for 24hr files
#     'annotations.mat'         # your .mat annotations file
# )


#month long HDF5 source file:
h5_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files\\' 
filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20140401T000000Z_20140501T000000Z-Ensemble300s_binMapNearest.h5'
#filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250422T000006Z_20250422T115956Z-binMapNearest.h5'
input_file = h5_folder + filename

#Output folder:
h5_24h_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\\' 

#.mat format annotations file:
annotations_file = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\annotations_table_ed05_revised.mat'

split_h5_to_24hr_files_with_ann(
    input_file,             # your month-long HDF5 source file (created with import_monthly_mat_to_h5)
    h5_24h_folder,          # output dir for 24hr files
    annotations_file,        # your .mat annotations file
)

Retaining static dataset: amplitudeVerticalRange shape (102,)
Retaining static dataset: corScreenThresholdUsed shape (0,)
Retaining static dataset: processingComments shape (9,)
Retaining static dataset: range shape (102,)
2012-06-02 00:00:00
2013-05-11 00:00:00
2014-04-01 00:00:00
2014-04-01 23:59:59
2013-06-19 05:29:01
2013-06-19 07:00:27
2014-04-01 00:00:00
2014-04-01 23:59:59
2013-06-19 11:15:00
2013-06-20 04:05:00
2014-04-01 00:00:00
2014-04-01 23:59:59
2013-07-06 04:59:32
2013-07-06 19:12:00
2014-04-01 00:00:00
2014-04-01 23:59:59
2014-08-29 20:35:00
2014-08-29 21:20:00
2014-04-01 00:00:00
2014-04-01 23:59:59
Wrote F:\\Documents\\GitHub\\ml_development\\ADCP_ML\\h5_24h_files\\20140401T000000_20140401T235959.h5 (len=288, 0 annotations)
Retaining static dataset: amplitudeVerticalRange shape (102,)
Retaining static dataset: corScreenThresholdUsed shape (0,)
Retaining static dataset: processingComments shape (9,)
Retaining static dataset: range shape (102,)
Wrote F:\\Documents\\GitHu

In [ ]:
# Add repo root to Python path - Needed to import from src folder
import sys
from pathlib import Path
repo_root = Path().resolve().parent  # notebooks/ → ADCP-CNN-QAQC
sys.path.append(str(repo_root))

#Verify that the code for making annotations is correct
from src.split_h5_to_24hr_files import load_matlab_annotations

#.mat format annotations file:
annotations_file = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\annotations_table_ed05_revised.mat'

annotations = load_matlab_annotations(annotations_file)

#This part appears to be correct..
annotations[31]


{'class': 'Dropout A',
 'startDate': '2024-07-01 22:30:00',
 'endDate': '2024-07-02 18:32:00',
 'comment': 'Beam 1',
 'start_datetime': datetime.datetime(2024, 7, 1, 22, 30, tzinfo=datetime.timezone.utc),
 'end_datetime': datetime.datetime(2024, 7, 2, 18, 32, tzinfo=datetime.timezone.utc)}

{'class': 'Dropout A',
 'startDate': '2013-06-19 11:15:00',
 'endDate': '2013-06-20 04:05:00',
 'comment': 'Beam 2',
 'start_datetime': datetime.datetime(2013, 6, 19, 11, 15, tzinfo=datetime.timezone.utc),
 'end_datetime': datetime.datetime(2013, 6, 20, 4, 5, tzinfo=datetime.timezone.utc)}

In [73]:
#STEP 3: accesss the output data, and make plots with it:

import h5py
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

def make_sanity_plots(h5file, outdir=None, show=True):
    import os
    time_series_vars = []
    time_series_data = []

    with h5py.File(h5file, 'r') as f:
        data_grp = f['/data']
        varnames = list(data_grp.keys())
        time = data_grp['time'][:]
        time_dt = np.array(time, dtype='datetime64[s]')
        range_data = data_grp['range'][:]
        # Generate an x-axis in hours since chunk start (for plotting)
        t_hours = (time - time[0]) / 3600

        # Load annotations
        annotations = f.get('annotations', None)
        ann_lines = []
        if annotations is not None and len(annotations) > 0:
            # marking start and end of annotation events
            for st, et in zip(annotations['start_time'], annotations['end_time']):
                idx_start = np.searchsorted(time, st)
                idx_end = np.searchsorted(time, et)
                ann_lines.append(idx_start)
                ann_lines.append(idx_end)

        for var in varnames:
            if var == "time":
                continue

            arr = data_grp[var][()]
            shape = arr.shape

            # ====== 1D time series check ======
            if arr.ndim == 1 and len(arr) == len(time):
                time_series_vars.append(var)
                time_series_data.append(arr)
                continue
            elif arr.ndim == 1:
                #Probably just range, no plots needed
                continue

            # ------ Multidimensional "image" data ------
            ndim = arr.ndim
            # Guess axes: time/range/channel
            time_dim = None
            channel_dim = None
            range_dim = None

            # time axis: size == len(time)
            for i, sz in enumerate(arr.shape):
                if sz == len(time):
                    time_dim = i
            # channel axis: size 2~5
            chans = [i for i, sz in enumerate(arr.shape) if 2 <= sz <= 5 and i != time_dim]
            channel_dim = chans[0] if chans else None
            # range axis: remaining
            rngs = [i for i in range(ndim) if i != time_dim and i != channel_dim]
            range_dim = rngs[0] if rngs else None

            # Reorder: time, range, channel (as available)
            axes_to_order = []
            axes_labels = []
            if time_dim is not None:
                axes_to_order.append(time_dim)
                axes_labels.append("time")
            if range_dim is not None:
                axes_to_order.append(range_dim)
                axes_labels.append("range")
            if channel_dim is not None:
                axes_to_order.append(channel_dim)
                axes_labels.append("channel")
            arrp = np.moveaxis(arr, axes_to_order, range(len(axes_to_order)))
            while arrp.ndim < 3:
                arrp = np.expand_dims(arrp, -1)
                axes_labels.append(f'axis{arrp.ndim-1}')

            n_channels = arrp.shape[2]
            fig, axs = plt.subplots(n_channels, 1, sharex=True, figsize=(12, 2.5*n_channels))
            if n_channels == 1:
                axs = [axs]

            # Create a mesh for the extent, using the actual times along x and range along y:
            extent = [
                mdates.date2num(time_dt[0].astype('M8[ms]').astype('O')),
                mdates.date2num(time_dt[-1].astype('M8[ms]').astype('O')),
                range_data[0], range_data[-1]
            ]

            for ch in range(n_channels):
                #Plot the Complex Data
                im = axs[ch].imshow(
                    arrp[:,:,ch].T, aspect='auto', origin='lower',
                    extent=[extent[0], extent[1], extent[2], extent[3]],
                    #extent=[t_hours[0], t_hours[-1], 0, arrp.shape[1]-1],
                    interpolation='nearest',
                    cmap='jet',
                )

                #Add labels and titles
                axs[ch].set_ylabel("Range bin" if range_dim is not None else '')
                axs[ch].set_title(f"{var} - Channel {ch+1}")

                #Add dashed vertical lines for annotations
                for vline in ann_lines:
                    if 0 <= vline < len(time_dt):
                        axs[ch].axvline(x=mdates.date2num(time_dt[vline]), color='cyan', linestyle='dashed', alpha=0.7)
                    #axs[ch].axvline(x=t_hours[vline], color='cyan', linestyle='dashed', alpha=0.7)
                        
                #Add a colorbar
                fig.colorbar(im, ax=axs[ch], label=var)
            
            # -- Date formatting for X --
            axs[-1].xaxis_date()  # tells matplotlib to interpret x as dates
            axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            fig.autofmt_xdate()  # Makes dates pretty (auto-rotates, etc.)

            axs[-1].set_xlabel(time_dt[0].astype('datetime64[D]').astype(str))   # 'yyyy-mm-dd' date for xlabel
            #axs[-1].set_xlabel("Time (hours since start)")
            fig.suptitle(f"{var} (shape={arr.shape})")
            plt.tight_layout()
            if outdir:
                if not os.path.exists(outdir):
                    os.makedirs(outdir)
                plt.savefig(f"{outdir}/{var}.png", dpi=120)
            if show:
                plt.show()
            plt.close()

    # === Time-series plots for all 1D fields ===
    if time_series_vars:
        n_series = len(time_series_vars)
        fig, axs = plt.subplots(n_series, 1, sharex=True, figsize=(14, 2.5*n_series))
        if n_series == 1:
            axs = [axs]
        for i, (var, arr) in enumerate(zip(time_series_vars, time_series_data)):
            axs[i].plot(t_hours, arr)
            axs[i].set_ylabel(var)
            for vline in ann_lines:
                axs[i].axvline(x=t_hours[vline], color='cyan', linestyle='dashed', alpha=0.7)
            axs[i].grid(True)
        axs[-1].set_xlabel("Time (hours since start)")
        plt.suptitle("Time series variables: " + ", ".join(time_series_vars))
        plt.tight_layout()
        if outdir:
            plt.savefig(f"{outdir}/time_series.png", dpi=120)
        if show:
            plt.show()
        plt.close()


In [ ]:
#Test code to extract and plot the data 
filename = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\20140401T000000_20140401T235959.h5'

make_sanity_plots(filename, outdir="./h5_24hr_figs", show=True)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'F:\\Documents\\GitHub\\ml_development\\ADCP_ML\\h5_24h_files\\20140401T000000_20140401T235959.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
#One final thing is to run this for a date where annotations exist

######################################
# Convert mat to h5 (still monthly format)
######################################

# Path to your .mat file

data_folder = r'F:\\Documents\\Projects\\ADCP\\scan_for_data\\BACAX\ADCP2MHZ\\20240401\\'
file_list = os.listdir(data_folder)
mat_files = {k for k in file_list if os.path.splitext(k)[1] == ".mat"}

print(mat_files)

mat_paths = []
for filename in mat_files:
    mat_paths.append(data_folder + filename) 

#data_folder = r'F:\\Documents\\Projects\\ADCP\\ADCP Monitoring Management\\20250422_BACAX_2MHZ\\'
#filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250422T000006Z_20250422T115956Z-binMapNearest.mat'
#mat_path = data_folder + filename

# Define output folder
output_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files'

#Run the extraction
for mat_path in mat_paths:
    extract_mat_to_h5(mat_path, output_folder) 

#extract_mat_to_h5(mat_path, output_folder) # For a single file
    

######################################
# Split to 24 hours, embed annotations
# and save the time in python format
######################################
    
#month long HDF5 source file:
h5_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files\\' 
filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20240401T000000Z_20240501T000000Z-Ensemble300s_binMapNearest.h5'
#filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250422T000006Z_20250422T115956Z-binMapNearest.h5'
input_file = h5_folder + filename

#Output folder:
h5_24h_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\\' 

#.mat format annotations file:
annotations_file = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\annotations_table_ed05_revised.mat'

split_h5_to_24hr_files_with_ann(
    input_file,             # your big HDF5 source
    h5_24h_folder,          # output dir for 24hr files
    annotations_file,        # your .mat annotations file
)


######################################
#Test code to extract and plot the data 
######################################

filename = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\\20240406T000000_20240406T235959.h5'

make_sanity_plots(filename, outdir="./h5_24hr_figs", show=True)

In [ ]:
######################################
# Split to 24 hours, embed annotations
# and save the time in python format
######################################
    
#month long HDF5 source file:
h5_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files\\' 
filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20240401T000000Z_20240501T000000Z-Ensemble300s_binMapNearest.h5'
#filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250422T000006Z_20250422T115956Z-binMapNearest.h5'
input_file = h5_folder + filename

#Output folder:
h5_24h_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\\' 

#.mat format annotations file:
annotations_file = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\annotations_table_ed05_revised.mat'

split_h5_to_24hr_files_with_ann(
    input_file,             # your big HDF5 source
    h5_24h_folder,          # output dir for 24hr files
    annotations_file,        # your .mat annotations file
)

Retaining static dataset: amplitudeVerticalRange shape (102,)
Retaining static dataset: corScreenThresholdUsed shape (0,)
Retaining static dataset: processingComments shape (9,)
Retaining static dataset: range shape (102,)


TypeError: Object dtype dtype('O') has no native HDF5 equivalent

In [ ]:
######################################
#Test code to extract and plot the data 
######################################
filename = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\\20240406T000000_20240406T235959.h5'

make_sanity_plots(filename, outdir="./h5_24hr_figs", show=True)

In [ ]:
#NEXT: I've created 2 python files:
# F:\Documents\GitHub\ml_development\ADCP_ML\convert_monthly_mat_to_h5.py
#F:\Documents\GitHub\ml_development\ADCP_ML\split_h5_to_24hr_files.py

#I'll want to create a loop that 

######################################
# Convert mat to h5 (still monthly format)
######################################

# Path to your .mat file

data_folder = r'F:\\Documents\\Projects\\ADCP\\scan_for_data\\BACAX\ADCP2MHZ\\20240401\\'
file_list = os.listdir(data_folder)
mat_files = {k for k in file_list if os.path.splitext(k)[1] == ".mat"}

print(mat_files)

mat_paths = []
for filename in mat_files:
    mat_paths.append(data_folder + filename) 

#data_folder = r'F:\\Documents\\Projects\\ADCP\\ADCP Monitoring Management\\20250422_BACAX_2MHZ\\'
#filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250422T000006Z_20250422T115956Z-binMapNearest.mat'
#mat_path = data_folder + filename

# Define output folder
output_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files'

#Run the extraction
for mat_path in mat_paths:
    extract_mat_to_h5(mat_path, output_folder) 

#extract_mat_to_h5(mat_path, output_folder) # For a single file
    

######################################
# Split to 24 hours, embed annotations
# and save the time in python format
######################################
    
#month long HDF5 source file:
h5_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files\\' 
filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20240401T000000Z_20240501T000000Z-Ensemble300s_binMapNearest.h5'
#filename = 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250422T000006Z_20250422T115956Z-binMapNearest.h5'
input_file = h5_folder + filename

#Output folder:
h5_24h_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\\' 

#.mat format annotations file:
annotations_file = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\annotations_table_ed05_revised.mat'

split_h5_to_24hr_files_with_ann(
    input_file,             # your big HDF5 source
    h5_24h_folder,          # output dir for 24hr files
    annotations_file,        # your .mat annotations file
)


######################################
#Test code to extract and plot the data 
######################################

filename = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files\20240406T000000_20240406T235959.h5'

make_sanity_plots(filename, outdir="./h5_24hr_figs", show=True)